# TP: Web Sémantique et SPARQL - Tom Belliver 
## Application à Ontologie African Wildlife

**Objectif :** Comprendre la structure des données en graphe (RDF) et apprendre à les interroger avec le langage SPARQL.

Ce TP progressif couvre:
1. Les **triplets RDF** (fondations)
2. Les **filtres et contraintes**
3. Les **données incomplètes** (OPTIONAL, UNION)
4. Les **agrégations** (COUNT, GROUP BY)
5. L'**ontologie African Wildlife** (application concrète)

## Partie 1: Les Fondations - Triplets RDF

### Concept: Qu'est-ce qu'un triplet RDF?

Contrairement aux bases de données relationnelles (avec tableaux), le **Web Sémantique** utilise des **graphes**.

L'unité de base est le **triplet** composé de trois éléments:

```
┌─────────────┬──────────────┬──────────────┐
│   SUJET     │   PRÉDICAT   │    OBJET     │
├─────────────┼──────────────┼──────────────┤
│ De qui?     │ Quel lien?   │ Quoi/Qui?    │
│ (Entité)    │ (Propriété)  │ (Valeur)     │
├─────────────┼──────────────┼──────────────┤
│  Simba      │   est un     │   Lion       │
│  Simba      │   habite à   │   Afrique    │
│  Lion       │   mange      │   Gazelle    │
└─────────────┴──────────────┴──────────────┘
```

**Exemple:** `Simba` → `est une espèce de` → `Lion`

En RDF, chaque partie est identifiée par une **URI** (adresse web unique) ou un **littéral** (valeur comme du texte ou un nombre).

### Installation et imports

In [1]:
# Installation de rdflib (décommenter si nécessaire)
#!pip install rdflib

from rdflib import Graph, Literal, URIRef, Namespace
from rdflib.namespace import RDF, RDFS, OWL, XSD


### Créer notre premier graphe: "Le Roi Lion"

Nous allons créer un petit graphe avec des animaux du Roi Lion et leurs propriétés.

In [2]:
# Création d'un graphe vide
g = Graph()

# Définition d'un espace de noms (namespace)
# C'est comme un préfixe qui permet de créer des URIs uniques
EX = Namespace("http://example.org/zoo/")

# Ajout de triplets au graphe
# Format: (Sujet, Prédicat, Objet)
g.add((EX.Simba, EX.species, Literal("Lion")))
g.add((EX.Simba, EX.name, Literal("Simba")))
g.add((EX.Timon, EX.species, Literal("Meerkat")))

print(f"Graphe créé avec {len(g)} triplets\n")
print("Triplets ajoutés:")
print("  1. Simba est un Lion")
print("  2. Simba s'appelle Simba")
print("  3. Timon est un Meerkat")

Graphe créé avec 3 triplets

Triplets ajoutés:
  1. Simba est un Lion
  2. Simba s'appelle Simba
  3. Timon est un Meerkat


### Ma première requête SPARQL

Maintenant, interrogeons notre graphe avec SPARQL. La structure de base est:

```sparql
SELECT ?variable1 ?variable2
WHERE {
    ?sujet ?predicat ?objet .
}
```

- `SELECT` : spécifie les variables à retourner
- `WHERE` : spécifie le motif à chercher
- Les `?` devant un nom indiquent une variable (à chercher)
- Le `.` termine chaque triplet

In [3]:
# Requête SPARQL basique
query = """
SELECT ?sujet ?espece
WHERE {
    ?sujet <http://example.org/zoo/species> ?espece .
}
"""

print("Résultats de la requête:")
for row in g.query(query):
    nom = str(row.sujet).split('/')[-1]
    espece = str(row.espece)
    print(f"{nom} est un {espece}")

Résultats de la requête:
Simba est un Lion
Timon est un Meerkat


### Exercice 1: À votre tour!

**Mission:** Ajouter deux nouveaux animaux au graphe et écrire une requête SPARQL pour afficher tous les noms d'animaux.


**Indice:** Utilisez le prédicat `EX.name` pour trouver les noms.

In [4]:
# TODO: Ajouter Nala (Lioness) et Pumbaa (Warthog)
g.add((EX.Nala, EX.species, Literal("Lioness")))
g.add((EX.Nala, EX.name, Literal("Nala")))
# Ajouter Pumbaa...

# TODO: Écrire une requête pour trouver tous les noms
query = """
SELECT ?nom
WHERE {
    # À compléter...
}
"""


---

## Partie 2: Filtrer et Préciser - Les Contraintes

### Concept: FILTER pour restreindre les résultats

Souvent, on ne veut pas *tous* les résultats, mais seulement ceux qui satisfont certaines conditions.

La clause `FILTER` en SPARQL est l'équivalent du `WHERE` conditionnel en SQL.

**Syntaxe:**
```sparql
SELECT ?variable
WHERE {
    ?sujet ?predicat ?variable .
    FILTER (?variable > 10)
}
```

**Comparateurs disponibles:** `>`, `<`, `>=`, `<=`, `=`, `!=`

### Ajouter des données typées (âges)

Ajoutons maintenant des âges à nos animaux. Les âges seront des **entiers** typés avec XSD (XML Schema Definition).

In [5]:
# Ajout de données avec type explicite
g.add((EX.Simba, EX.age, Literal(5, datatype=XSD.integer)))
g.add((EX.Timon, EX.age, Literal(12, datatype=XSD.integer)))
g.add((EX.Nala, EX.age, Literal(4, datatype=XSD.integer)))

print("Âges ajoutés:")
print("  Simba: 5 ans")
print("  Timon: 12 ans")
print("  Nala: 4 ans")

Âges ajoutés:
  Simba: 5 ans
  Timon: 12 ans
  Nala: 4 ans


### Requête avec FILTER

Cherchons maintenant tous les animaux qui ont plus de 4 ans.

In [6]:
query_filter = """
PREFIX zoo: <http://example.org/zoo/>

SELECT ?nom ?age
WHERE {
    ?x zoo:name ?nom .
    ?x zoo:age ?age .
    FILTER (?age > 4)
}
ORDER BY DESC(?age)
"""

print("Animaux ayant plus de 4 ans:")
for row in g.query(query_filter):
    print(f"{row.nom} a {row.age} ans")

Animaux ayant plus de 4 ans:
Simba a 5 ans


### Filtrer sur des chaînes de caractères

On peut aussi filtrer sur des **textes** avec des fonctions comme `CONTAINS`, `STRSTARTS`, `STRENDS`.

In [7]:
query_text = """
PREFIX zoo: <http://example.org/zoo/>

SELECT ?nom
WHERE {
    ?x zoo:name ?nom .
    FILTER (CONTAINS(lcase(str(?nom)), "a"))
}
ORDER BY ?nom
"""

print("Animaux dont le nom contient 'a' ou 'A':")
for row in g.query(query_text):
    print(row.nom)

Animaux dont le nom contient 'a' ou 'A':
Nala
Simba


### Exercice 2: Filtrer les prédateurs

**Mission:** Écrire une requête SPARQL qui affiche les animaux âgés de 5 à 10 ans (inclus).

**Indice:** Vous pouvez combiner plusieurs FILTER avec `&&` (ET) ou `||` (OU).

In [8]:
## TODO
query = """
SELECT ?nom ?age
WHERE {
    ?x zoo:name ?nom .
    ?x zoo:age ?age .
    FILTER (?age >= 5 && ?age <= 10)
}
"""

---

## Partie 3: Données Incomplètes et Chaînage - OPTIONAL & UNION

### Concept: Open World Assumption

Dans le Web Sémantique, on suppose qu'**on ne connaît pas tout** (Open World Assumption).

Si un animal n'a pas de surnom défini, ce n'est pas qu'il n'en a pas, c'est juste qu'on ne le sait pas.

**Problème:** Si on cherche `?x zoo:surnom ?surnom` dans le `WHERE`, on va perdre tous les animaux sans surnom.

**Solution:** Utiliser `OPTIONAL` pour dire "ce motif est optionnel".

### Ajouter des surnoms (données incomplètes)

In [9]:
# Seulement Simba a un surnom
g.add((EX.Simba, EX.nickname, Literal("King")))
# Timon et Nala n'ont pas de surnom

<Graph identifier=N9059002e0858477694ec394798532782 (<class 'rdflib.graph.Graph'>)>

### Requête OPTIONAL

Cherchons maintenant tous les animaux avec leurs surnoms, s'ils en ont.

In [10]:
query_optional = """
PREFIX zoo: <http://example.org/zoo/>

SELECT ?nom ?surnom
WHERE {
    ?x zoo:name ?nom .
    OPTIONAL { ?x zoo:nickname ?surnom }
}
ORDER BY ?nom
"""

print("Animaux avec leurs surnoms (si disponibles):")
for row in g.query(query_optional):
    surnom = str(row.surnom) if row.surnom else "Pas de surnom"
    print(f"{row.nom}: {surnom}")

Animaux avec leurs surnoms (si disponibles):
Nala: Pas de surnom
Simba: King


### Concept: UNION (motifs alternatifs)

Parfois, on veut chercher des données selon **plusieurs motifs différents**.

`UNION` permet de combiner deux motifs avec `{motif1} UNION {motif2}`.

In [11]:
query_union = """
PREFIX zoo: <http://example.org/zoo/>

SELECT ?entity ?info ?type
WHERE {
    {
        ?entity zoo:species ?info .
        BIND("espèce" AS ?type)
    }
    UNION
    {
        ?entity zoo:nickname ?info .
        BIND("surnom" AS ?type)
    }
}
ORDER BY ?entity ?type
"""

print("Espèces et surnoms (toutes les infos):")
for row in g.query(query_union):
    entity = str(row.entity).split('/')[-1]
    if row.type == "espèce":
        print(f"{entity} est un {row.info}")
    else:
        print(f"{entity} a le surnom: {row.info}")

Espèces et surnoms (toutes les infos):
Nala a le surnom: Lioness
Simba a le surnom: Lion
Simba a le surnom: King
Timon a le surnom: Meerkat


### Exercice 3: Combiner OPTIONAL et FILTER

**Mission:** Écrire une requête qui affiche tous les animaux de plus de 5 ans avec leurs surnoms (s'ils en ont).


In [12]:
query = """
SELECT ?nom ?surnom ?age
WHERE {
    # À compléter...
    OPTIONAL { ?x zoo:nickname ?surnom }
    FILTER (?age > 5)
}
"""

---

## Partie 4: Analyse de Données - Agrégations

### Concept: Compter, grouper, trier

SPARQL permet de faire des **statistiques** sur les données:
- `COUNT(?)` : compter les résultats
- `SUM(?)`, `AVG(?)`, `MIN(?)`, `MAX(?)` : statistiques
- `GROUP BY` : grouper par critère
- `ORDER BY` : trier les résultats
- `LIMIT` : limiter le nombre de résultats
- `DISTINCT` : éliminer les doublons

### Ajouter plus de données

In [13]:
# Ajouter Nala et d'autres animaux
g.add((EX.Nala, EX.species, Literal("Lioness")))
g.add((EX.Nala, EX.name, Literal("Nala")))
g.add((EX.Nala, EX.age, Literal(4, datatype=XSD.integer)))

g.add((EX.Mufasa, EX.species, Literal("Lion")))
g.add((EX.Mufasa, EX.name, Literal("Mufasa")))
g.add((EX.Mufasa, EX.age, Literal(15, datatype=XSD.integer)))

g.add((EX.Pumbaa, EX.species, Literal("Warthog")))
g.add((EX.Pumbaa, EX.name, Literal("Pumbaa")))
g.add((EX.Pumbaa, EX.age, Literal(8, datatype=XSD.integer)))

g.add((EX.Zazu, EX.species, Literal("Hornbill")))
g.add((EX.Zazu, EX.name, Literal("Zazu")))
g.add((EX.Zazu, EX.age, Literal(13, datatype=XSD.integer)))

print("Données étendues créées")
print("  - Mufasa (Lion, 15 ans)")
print("  - Pumbaa (Warthog, 8 ans)")
print("  - Zazu (Hornbill, 13 ans)")

Données étendues créées
  - Mufasa (Lion, 15 ans)
  - Pumbaa (Warthog, 8 ans)
  - Zazu (Hornbill, 13 ans)


### Requête avec agrégation: COUNT et GROUP BY

Comptons combien d'animaux il y a par espèce.

In [14]:
query_agg = """
PREFIX zoo: <http://example.org/zoo/>

SELECT ?espece (COUNT(?x) as ?total) (AVG(?age) as ?age_moyen)
WHERE {
    ?x zoo:species ?espece .
    ?x zoo:age ?age .
}
GROUP BY ?espece
ORDER BY DESC(?total)
"""

print("Nombre d'animaux par espèce:")
for row in g.query(query_agg):
    age_moyen = float(row.age_moyen) if row.age_moyen else 0
    print(f"{row.espece}: {row.total} individu(s), âge moyen: {age_moyen:.1f} ans")

Nombre d'animaux par espèce:
Lion: 2 individu(s), âge moyen: 10.0 ans
Meerkat: 1 individu(s), âge moyen: 12.0 ans
Lioness: 1 individu(s), âge moyen: 4.0 ans
Warthog: 1 individu(s), âge moyen: 8.0 ans
Hornbill: 1 individu(s), âge moyen: 13.0 ans


### LIMIT et DISTINCT

- `LIMIT n` : retourne au maximum n résultats
- `DISTINCT` : élimine les doublons

In [15]:
query_limit = """
PREFIX zoo: <http://example.org/zoo/>

SELECT ?nom ?age
WHERE {
    ?x zoo:name ?nom .
    ?x zoo:age ?age .
}
ORDER BY DESC(?age)
LIMIT 3
"""

print("Les 3 animaux les plus âgés:")
for row in g.query(query_limit):
    print(f"{row.nom}: {row.age} ans")

Les 3 animaux les plus âgés:
Mufasa: 15 ans
Zazu: 13 ans
Pumbaa: 8 ans


### Exercice 4: Statistiques avancées

**Mission:** Trouver l'animal le plus âgé de chaque espèce.

**Indice:** Combinez `GROUP BY`, `MAX()`, et `?espece`.

In [16]:
## TODO

---

## Partie 5: Application Concrète - African Wildlife Ontology

### Introduction à l'ontologie African Wildlife

L'**ontologie African Wildlife** est une ontologie réelle créée par **C. Maria Keet** qui modélise:
- Les **classes** (animaux, plantes, habitats)
- Les **relations** (mange, vit dans, est une partie de)
- Les **restrictions** (un Lion est quelque chose qui mange des Herbivores)

C'est une ontologie en **OWL** (Web Ontology Language), qui est une extension de RDF.

### Charger l'ontologie

In [17]:
# Créer un nouveau graphe pour l'ontologie
aw_graph = Graph()

# Définir les namespaces
AW = Namespace("http://www.meteck.org/files/ontologies/AfricanWildlifeOntology1.owl#")

# Charger l'ontologie (on va créer une version simplifiée pour la démonstration)
print("Création d'une ontologie de démonstration (African Wildlife)...")

# Classes principales
aw_graph.add((AW.Animal, RDF.type, OWL.Class))
aw_graph.add((AW.Mammal, RDF.type, OWL.Class))
aw_graph.add((AW.Carnivore, RDF.type, OWL.Class))
aw_graph.add((AW.Herbivore, RDF.type, OWL.Class))
aw_graph.add((AW.Lion, RDF.type, OWL.Class))
aw_graph.add((AW.Gazelle, RDF.type, OWL.Class))
aw_graph.add((AW.Elephant, RDF.type, OWL.Class))
aw_graph.add((AW.Zebra, RDF.type, OWL.Class))
aw_graph.add((AW.Plant, RDF.type, OWL.Class))
aw_graph.add((AW.Grass, RDF.type, OWL.Class))

# Hiérarchie des classes
aw_graph.add((AW.Lion, RDFS.subClassOf, AW.Mammal))
aw_graph.add((AW.Lion, RDFS.subClassOf, AW.Carnivore))
aw_graph.add((AW.Gazelle, RDFS.subClassOf, AW.Mammal))
aw_graph.add((AW.Gazelle, RDFS.subClassOf, AW.Herbivore))
aw_graph.add((AW.Elephant, RDFS.subClassOf, AW.Mammal))
aw_graph.add((AW.Elephant, RDFS.subClassOf, AW.Herbivore))
aw_graph.add((AW.Zebra, RDFS.subClassOf, AW.Mammal))
aw_graph.add((AW.Zebra, RDFS.subClassOf, AW.Herbivore))
aw_graph.add((AW.Grass, RDFS.subClassOf, AW.Plant))
aw_graph.add((AW.Mammal, RDFS.subClassOf, AW.Animal))
aw_graph.add((AW.Carnivore, RDFS.subClassOf, AW.Animal))
aw_graph.add((AW.Herbivore, RDFS.subClassOf, AW.Animal))

# Propriétés (relations)
eats = Namespace("http://www.meteck.org/files/ontologies/AfricanWildlifeOntology1.owl#eats")
hunts = Namespace("http://www.meteck.org/files/ontologies/AfricanWildlifeOntology1.owl#hunts")
eats_prop = URIRef("http://www.meteck.org/files/ontologies/AfricanWildlifeOntology1.owl#eats")
hunts_prop = URIRef("http://www.meteck.org/files/ontologies/AfricanWildlifeOntology1.owl#hunts")

# Quelques relations
aw_graph.add((AW.Lion, eats_prop, AW.Gazelle))
aw_graph.add((AW.Lion, hunts_prop, AW.Zebra))
aw_graph.add((AW.Gazelle, eats_prop, AW.Grass))
aw_graph.add((AW.Elephant, eats_prop, AW.Grass))
aw_graph.add((AW.Zebra, eats_prop, AW.Grass))

print(f"Ontologie chargée avec {len(aw_graph)} triplets")

Création d'une ontologie de démonstration (African Wildlife)...
Ontologie chargée avec 27 triplets


### Explorer les classes de l'ontologie

Listez toutes les classes pour comprendre la structure.

In [18]:
query_classes = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT DISTINCT ?classe
WHERE {
    ?classe rdf:type owl:Class .
}
"""

print("Classes trouvées dans l'ontologie African Wildlife:")
classes = []
for i, row in enumerate(aw_graph.query(query_classes), 1):
    classe_name = str(row.classe).split('#')[-1]
    classes.append(classe_name)
    print(f"{i}. {classe_name}")

Classes trouvées dans l'ontologie African Wildlife:
1. Animal
2. Mammal
3. Carnivore
4. Herbivore
5. Lion
6. Gazelle
7. Elephant
8. Zebra
9. Plant
10. Grass


### Explorer la hiérarchie des classes

Voyons la relation `rdfs:subClassOf` qui définit la hiérarchie.

In [19]:
query_hierarchy = """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?parent ?enfant
WHERE {
    ?enfant rdfs:subClassOf ?parent .
}
"""

print("Hiérarchie des classes (parent ← enfant):")
for row in aw_graph.query(query_hierarchy):
    parent_name = str(row.parent).split('#')[-1]
    enfant_name = str(row.enfant).split('#')[-1]
    print(f"{parent_name} ← {enfant_name}")

Hiérarchie des classes (parent ← enfant):
Mammal ← Lion
Mammal ← Gazelle
Mammal ← Elephant
Mammal ← Zebra
Carnivore ← Lion
Herbivore ← Gazelle
Herbivore ← Elephant
Herbivore ← Zebra
Plant ← Grass
Animal ← Mammal
Animal ← Carnivore
Animal ← Herbivore


### Chaîne alimentaire: Qui mange qui?

Explorez les relations mange pour comprendre la chaîne alimentaire.

In [20]:
query_food_chain = """
PREFIX aw: <http://www.meteck.org/files/ontologies/AfricanWildlifeOntology1.owl#>

SELECT ?predateur ?nourriture ?relation
WHERE {
    {
        ?predateur aw:eats ?nourriture .
        BIND("mange" AS ?relation)
    }
    UNION
    {
        ?predateur aw:hunts ?nourriture .
        BIND("chasse" AS ?relation)
    }
}
ORDER BY ?predateur
"""

print("Chaîne alimentaire (qui mange quoi):")
for row in aw_graph.query(query_food_chain):
    pred = str(row.predateur).split('#')[-1]
    nour = str(row.nourriture).split('#')[-1]
    rel = str(row.relation)
    print(f"{pred} {rel} {nour}")

Chaîne alimentaire (qui mange quoi):
Elephant mange Grass
Gazelle mange Grass
Lion mange Gazelle
Lion chasse Zebra
Zebra mange Grass


### Carnivores vs Herbivores

Séparons les animaux par régime alimentaire.

In [21]:
query_diet = """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX aw: <http://www.meteck.org/files/ontologies/AfricanWildlifeOntology1.owl#>

SELECT ?type (GROUP_CONCAT(?animal; separator=", ") AS ?animaux)
WHERE {
    {
        ?animal rdfs:subClassOf aw:Carnivore .
        BIND("Carnivore" AS ?type)
    }
    UNION
    {
        ?animal rdfs:subClassOf aw:Herbivore .
        BIND("Herbivore" AS ?type)
    }
}
GROUP BY ?type
"""

print("Carnivores (animaux qui chassent/mangent d'autres animaux):")
for row in aw_graph.query(query_diet):
    if "Carnivore" in str(row.type):
        animaux = str(row.animaux).split(',') if row.animaux else []
        for a in animaux:
            if a.strip():
                print(a.strip().split('#')[-1])

print("\nHerbivores (animaux qui mangent des plantes):")
for row in aw_graph.query(query_diet):
    if "Herbivore" in str(row.type):
        animaux = str(row.animaux).split(',') if row.animaux else []
        for a in animaux:
            if a.strip():
                print(a.strip().split('#')[-1])

Carnivores (animaux qui chassent/mangent d'autres animaux):
Lion

Herbivores (animaux qui mangent des plantes):
Gazelle
Elephant
Zebra


### Implémentation d'un raisonneur

Le principal intérêt des ontologies sont les raisonneurs : ils permettent de vérifier qu'il n'y a pas d'erreur dans l'onthologie. Par exemple, un animal ne peut pas être carnivore et herbivore à la fois. 

In [ ]:
# si nécessaire
# !pip install rdflib owlrl 
from rdflib import Graph, Namespace, RDF
from owlrl import DeductiveClosure, OWLRL_Semantics

AW = Namespace("http://example.org/african-wildlife#")

g = Graph()

g.parse(data="""
@prefix aw: <http://example.org/african-wildlife#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

aw:Animal a owl:Class .
aw:Carnivore a owl:Class .
aw:Herbivore a owl:Class .
aw:Zebra a owl:Class ;
    rdfs:subClassOf aw:Herbivore .

aw:Carnivore owl:disjointWith aw:Herbivore .
""", format="turtle")


g.add((AW.ZebraX, RDF.type, AW.Zebra))
g.add((AW.ZebraX, RDF.type, AW.Carnivore))

print("Avant raisonnement :", len(g))
print((AW.ZebraX, RDF.type, AW.Herbivore) in g)

DeductiveClosure(OWLRL_Semantics).expand(g)

print("Après raisonnement :", len(g))
print((AW.ZebraX, RDF.type, AW.Herbivore) in g)

print("On peut donc vérifier si un zèbre est carnivore dans la base, et corriger cette dernière.")

Avant raisonnement : 8
False
Après raisonnement : 140
True
On peut donc vérifier qusi un zèbre est carnivore dans la base, et corriger cette dernière.


### Défi Final: Requête Personnalisée

**Mission:** Écrire une requête SPARQL qui:
1. Trouve tous les **carnivores** de l'ontologie
2. Pour chaque carnivore, liste ses **proies** (ce qu'il mange/chasse)
3. Affiche les résultats de manière lisible
4. Utilise un raisonneur pour vérifier que les herbivores ne sont pas carnivores

**Indices:**
- Utilisez `rdfs:subClassOf` pour trouver les carnivores
- Utilisez `aw:eats` et `aw:hunts` pour les relations alimentaires
- Pensez à utiliser `UNION` si les propriétés varient


**À vous de jouer!**

In [22]:
## TODO

query_defi = """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX aw: <http://www.meteck.org/files/ontologies/AfricanWildlifeOntology1.owl#>

SELECT ?carnivore ?proie
WHERE {
    # TODO: Trouver les carnivores
    ?carnivore rdfs:subClassOf aw:Carnivore .
    # TODO: Trouver les proies
    {
        ?carnivore aw:eats ?proie .
    }
    UNION
    {
        ?carnivore aw:hunts ?proie .
    }
}
ORDER BY ?carnivore
"""



### Ressources utiles:

- [SPARQL Query Language Specification](https://www.w3.org/TR/sparql11-query/)
- [RDFlib Documentation](https://rdflib.readthedocs.io/)
- [African Wildlife Ontology](https://www.meteck.org/)
- [Wikidata Query Service](https://query.wikidata.org/)
- [DBpedia SPARQL Endpoint](https://dbpedia.org/sparql)